# 📖 Notebook 2: TCP vs UDP Deep Dive

📖 **Source**: [Hello Interview – Networking Essentials](https://www.hellointerview.com/learn/system-design/core-concepts/networking-essentials)

Every time two computers talk over a network, they use a **transport layer protocol** — either TCP or UDP. Think of TCP as sending a registered letter (guaranteed delivery, in order) and UDP as shouting across a room (fast, but some words might get lost).

## Learning Objectives

By the end of this notebook, you'll understand:
- How TCP provides reliable, ordered delivery (and the cost of doing so)
- How UDP provides fast, connectionless communication (and what you give up)
- How to write a simple TCP and UDP server/client with Python sockets
- When to choose each protocol in system design

## 🛠️ Setup

This notebook uses only Python's built-in `socket` module — **no Docker needed**.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import socket
import threading
import time

print("✅ All imports successful — let's explore TCP and UDP!")

---
## Part 1: TCP — Reliable and Ordered

TCP (Transmission Control Protocol) is the **workhorse of the internet**. When you load a web page, send an email, or download a file, TCP ensures every byte arrives correctly and in order.

### How TCP Establishes a Connection: The 3-Way Handshake

Before any data flows, TCP requires both sides to agree to communicate:

```
Client                    Server
  │                         │
  │──── SYN ───────────────→│  "Hey, I want to connect"
  │                         │
  │←──── SYN-ACK ──────────│  "OK, I heard you, let's connect"
  │                         │
  │──── ACK ───────────────→│  "Great, connection established!"
  │                         │
  │←───── Data flows ──────→│
```

This takes **1.5 round trips** before any data can be sent. That's the price of reliability.

### Key Characteristics of TCP
- **Connection-oriented**: Must establish a connection first (3-way handshake)
- **Reliable delivery**: Guarantees data arrives in order, without errors
- **Flow control**: Prevents overwhelming the receiver
- **Congestion control**: Adapts to network conditions

Let's build a TCP server and client to see it work!

In [ ]:
# === TCP Echo Server ===
# This server waits for a client to connect, reads a message,
# and sends it back (echoes it).

def tcp_echo_server(host="127.0.0.1", port=9001):
    """A simple TCP server that echoes back whatever it receives."""
    # SOCK_STREAM = TCP (a stream of bytes, reliable and ordered)
    server_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_sock.bind((host, port))
    server_sock.listen(1)  # accept at most 1 pending connection
    server_sock.settimeout(10)  # don't wait forever

    print(f"[TCP Server] Listening on {host}:{port}...")

    try:
        # Step 1: Accept a connection (this is where the 3-way handshake happens)
        conn, addr = server_sock.accept()
        print(f"[TCP Server] Client connected from {addr}")

        # Step 2: Receive data
        data = conn.recv(1024)  # read up to 1024 bytes
        print(f"[TCP Server] Received: {data.decode()}")

        # Step 3: Send data back (echo)
        response = f"ECHO: {data.decode()}"
        conn.sendall(response.encode())
        print(f"[TCP Server] Sent: {response}")

        conn.close()
    finally:
        server_sock.close()
        print("[TCP Server] Shut down.")


def tcp_echo_client(message, host="127.0.0.1", port=9001):
    """A simple TCP client that sends a message and prints the response."""
    time.sleep(0.3)  # give the server a moment to start

    client_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    # Step 1: Connect (3-way handshake happens here)
    print(f"[TCP Client] Connecting to {host}:{port}...")
    client_sock.connect((host, port))
    print(f"[TCP Client] Connected!")

    # Step 2: Send data
    client_sock.sendall(message.encode())
    print(f"[TCP Client] Sent: {message}")

    # Step 3: Receive response
    response = client_sock.recv(1024)
    print(f"[TCP Client] Received: {response.decode()}")

    client_sock.close()


# Run server and client in separate threads
server_thread = threading.Thread(target=tcp_echo_server)
client_thread = threading.Thread(target=tcp_echo_client, args=("Hello, TCP!",))

server_thread.start()
client_thread.start()

server_thread.join()
client_thread.join()

print("\n💡 TCP guarantees: the client WILL get a response, and it WILL be complete.")

### TCP Guarantees: Ordering

TCP guarantees that messages arrive **in order**. Even if network packets arrive out of order, TCP reassembles them before delivering to your application.

In [ ]:
# === TCP Ordering Demo ===
# Send numbered messages and verify they arrive in order.

received_messages = []

def tcp_ordering_server(host="127.0.0.1", port=9002):
    server_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_sock.bind((host, port))
    server_sock.listen(1)
    server_sock.settimeout(10)

    conn, _ = server_sock.accept()
    # Receive all messages (they arrive as a stream of bytes)
    data = b""
    while True:
        chunk = conn.recv(4096)
        if not chunk:
            break
        data += chunk

    for line in data.decode().strip().split("\n"):
        received_messages.append(line)

    conn.close()
    server_sock.close()


def tcp_ordering_client(host="127.0.0.1", port=9002):
    time.sleep(0.3)
    client_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client_sock.connect((host, port))

    # Send 20 numbered messages
    for i in range(20):
        client_sock.sendall(f"Message {i}\n".encode())

    client_sock.close()


server_thread = threading.Thread(target=tcp_ordering_server)
client_thread = threading.Thread(target=tcp_ordering_client)

server_thread.start()
client_thread.start()
server_thread.join()
client_thread.join()

print("Messages received by TCP server (in order):")
for msg in received_messages:
    print(f"  {msg}")

# Verify ordering
expected = [f"Message {i}" for i in range(20)]
if received_messages == expected:
    print("\n✅ All 20 messages arrived in perfect order — that's TCP!")
else:
    print("\n❌ Messages out of order (shouldn't happen with TCP!)")

---
## Part 2: UDP — Fast but Unreliable

UDP (User Datagram Protocol) is the opposite philosophy from TCP. It's like the **machine gun** of protocols: spray and pray.

```
Client                    Server
  │                         │
  │──── Data ──────────────→│  "Here's some data" (no handshake!)
  │──── Data ──────────────→│  "Here's more data"
  │──── Data ───────X       │  (this one got lost, nobody knows)
  │──── Data ──────────────→│  "And more data"
```

### Key Characteristics of UDP
- **Connectionless**: No handshake, just start sending
- **No guarantee of delivery**: Packets may be lost silently
- **No ordering**: Packets may arrive in a different order
- **Very fast**: Minimal overhead (only 8 bytes of header vs TCP's 20-60)

In [ ]:
# === UDP Echo Server ===
# Notice: NO connection setup, NO accept(), just recv and send.

def udp_echo_server(host="127.0.0.1", port=9003):
    """A simple UDP server that echoes back whatever it receives."""
    # SOCK_DGRAM = UDP (individual datagrams, not a stream)
    server_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    server_sock.bind((host, port))
    server_sock.settimeout(5)

    print(f"[UDP Server] Listening on {host}:{port}...")

    try:
        # No accept() needed! Just receive data from anyone.
        data, addr = server_sock.recvfrom(1024)
        print(f"[UDP Server] Received from {addr}: {data.decode()}")

        # Send response back to the sender
        response = f"ECHO: {data.decode()}"
        server_sock.sendto(response.encode(), addr)
        print(f"[UDP Server] Sent: {response}")
    finally:
        server_sock.close()
        print("[UDP Server] Shut down.")


def udp_echo_client(message, host="127.0.0.1", port=9003):
    """A simple UDP client that sends a message and prints the response."""
    time.sleep(0.3)

    client_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    client_sock.settimeout(5)

    # No connect() needed! Just send to the address.
    print(f"[UDP Client] Sending to {host}:{port}...")
    client_sock.sendto(message.encode(), (host, port))
    print(f"[UDP Client] Sent: {message}")

    # Wait for response
    data, addr = client_sock.recvfrom(1024)
    print(f"[UDP Client] Received: {data.decode()}")

    client_sock.close()


# Run server and client
server_thread = threading.Thread(target=udp_echo_server)
client_thread = threading.Thread(target=udp_echo_client, args=("Hello, UDP!",))

server_thread.start()
client_thread.start()

server_thread.join()
client_thread.join()

print("\n💡 Notice: no handshake, no connect(). Just fire and receive.")
print("   UDP is simpler but there's no guarantee the message arrived.")

---
## Part 3: TCP vs UDP — Side by Side

Let's measure the difference in speed between TCP and UDP for sending many small messages.

In [ ]:
# === Performance Comparison: TCP vs UDP ===

NUM_MESSAGES = 1000
MESSAGE = b"Hello!"  # small message

# --- TCP Performance ---
def tcp_perf_server(port=9004):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("127.0.0.1", port))
    s.listen(1)
    s.settimeout(15)
    conn, _ = s.accept()
    count = 0
    while True:
        data = conn.recv(1024)
        if not data:
            break
        count += 1
    conn.close()
    s.close()
    return count

tcp_count = [0]

def tcp_perf_server_wrapper():
    tcp_count[0] = tcp_perf_server()

server_thread = threading.Thread(target=tcp_perf_server_wrapper)
server_thread.start()
time.sleep(0.3)

# TCP Client: connect once, send many messages
client_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client_sock.connect(("127.0.0.1", 9004))

tcp_start = time.perf_counter()
for _ in range(NUM_MESSAGES):
    client_sock.sendall(MESSAGE)
client_sock.close()
tcp_time = time.perf_counter() - tcp_start

server_thread.join()

# --- UDP Performance ---
udp_received = [0]

def udp_perf_server(port=9005):
    s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    s.bind(("127.0.0.1", port))
    s.settimeout(2)
    count = 0
    try:
        while True:
            s.recvfrom(1024)
            count += 1
    except socket.timeout:
        pass
    s.close()
    udp_received[0] = count

server_thread = threading.Thread(target=udp_perf_server)
server_thread.start()
time.sleep(0.3)

# UDP Client: no connection, just blast messages
client_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

udp_start = time.perf_counter()
for _ in range(NUM_MESSAGES):
    client_sock.sendto(MESSAGE, ("127.0.0.1", 9005))
client_sock.close()
udp_time = time.perf_counter() - udp_start

server_thread.join()

# Results
print(f"Sent {NUM_MESSAGES} messages with each protocol:\n")
print(f"  TCP: {tcp_time*1000:.1f} ms (all {NUM_MESSAGES} delivered guaranteed)")
print(f"  UDP: {udp_time*1000:.1f} ms ({udp_received[0]}/{NUM_MESSAGES} received)")
print(f"\n  UDP is ~{tcp_time/udp_time:.1f}× faster (on localhost)")

if udp_received[0] < NUM_MESSAGES:
    lost = NUM_MESSAGES - udp_received[0]
    print(f"\n⚠️  UDP lost {lost} packets! On a real network, this would be worse.")
else:
    print("\n💡 On localhost, UDP rarely loses packets. On a real network, it would.")

---
## Part 4: Connection Overhead — TCP's Hidden Cost

Every new TCP connection requires a 3-way handshake. If you make lots of short-lived connections, this overhead adds up.

In [ ]:
# === TCP Connection Overhead ===
# Compare: 100 connections (1 message each) vs 1 connection (100 messages)

def tcp_server_multi(port, num_connections):
    """Accept multiple connections, read one message each."""
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("127.0.0.1", port))
    s.listen(5)
    s.settimeout(15)
    for _ in range(num_connections):
        conn, _ = s.accept()
        conn.recv(1024)
        conn.close()
    s.close()

NUM_CONNS = 100

# Approach 1: Many connections, 1 message each
server_thread = threading.Thread(target=tcp_server_multi, args=(9006, NUM_CONNS))
server_thread.start()
time.sleep(0.3)

start = time.perf_counter()
for _ in range(NUM_CONNS):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect(("127.0.0.1", 9006))
    s.sendall(b"msg")
    s.close()
many_conn_time = time.perf_counter() - start
server_thread.join()

# Approach 2: 1 connection, many messages
def tcp_server_single(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.bind(("127.0.0.1", port))
    s.listen(1)
    s.settimeout(15)
    conn, _ = s.accept()
    while True:
        data = conn.recv(1024)
        if not data:
            break
    conn.close()
    s.close()

server_thread = threading.Thread(target=tcp_server_single, args=(9007,))
server_thread.start()
time.sleep(0.3)

s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
s.connect(("127.0.0.1", 9007))

start = time.perf_counter()
for _ in range(NUM_CONNS):
    s.sendall(b"msg")
s.close()
single_conn_time = time.perf_counter() - start
server_thread.join()

print(f"Sending {NUM_CONNS} messages:\n")
print(f"  {NUM_CONNS} connections (1 msg each): {many_conn_time*1000:.1f} ms")
print(f"  1 connection ({NUM_CONNS} msgs):       {single_conn_time*1000:.1f} ms")
print(f"\n  Reusing one connection is ~{many_conn_time/single_conn_time:.0f}× faster!")
print("\n💡 This is why HTTP keep-alive and HTTP/2 multiplexing exist —")
print("   they reuse connections to avoid the handshake overhead.")

---
## Part 5: When to Choose Each Protocol

| Use Case | Protocol | Why |
|----------|----------|-----|
| Web pages (HTTP) | **TCP** | Every byte of HTML/CSS/JS must arrive correctly |
| File downloads | **TCP** | Can't have missing chunks in a file |
| Database queries | **TCP** | Data integrity is critical |
| Video streaming | **UDP** | A dropped frame is better than a delayed stream |
| Online gaming | **UDP** | Low latency > perfect data (old position data is useless) |
| VoIP / voice calls | **UDP** | A tiny audio glitch beats a frozen call |
| DNS lookups | **UDP** | Simple question/answer, retry if lost |
| IoT telemetry | **UDP** | High volume, occasional loss is fine |

### Interview Rule of Thumb

> **Default to TCP** unless you're designing for real-time media (video/audio/gaming) where low latency matters more than perfect delivery.

Most interviewers expect TCP by default — they'll be impressed if you can explain *when and why* to switch to UDP.

In [ ]:
# === Quick Reference: TCP vs UDP ===

comparison = {
    "Feature":          ["Connection",   "Reliability",      "Ordering",           "Speed",    "Header Size", "Use Cases"],
    "TCP":              ["Required",     "Guaranteed",       "Guaranteed",         "Slower",   "20-60 bytes", "Web, APIs, DBs"],
    "UDP":              ["Not needed",   "Best-effort",      "No guarantee",       "Faster",   "8 bytes",     "Streaming, gaming"],
}

# Print a nice comparison table
print(f"{'Feature':<20} {'TCP':<20} {'UDP':<20}")
print("─" * 60)
for i, feature in enumerate(comparison["Feature"]):
    tcp_val = comparison["TCP"][i]
    udp_val = comparison["UDP"][i]
    print(f"{feature:<20} {tcp_val:<20} {udp_val:<20}")

---
## 🎓 Key Takeaways

1. **TCP** provides reliable, ordered delivery at the cost of connection setup and overhead
2. **UDP** is fast and lightweight but doesn't guarantee delivery or ordering
3. **TCP's 3-way handshake** adds latency — connection reuse (keep-alive) helps
4. Most internet traffic uses **TCP** — it's the safe default
5. Use **UDP** for real-time media where speed matters more than reliability
6. Modern protocols like **QUIC** (used by HTTP/3) aim to combine TCP's reliability with UDP's speed

### Interview Tips
- Assume **TCP** unless the problem involves streaming, gaming, or VoIP
- If you suggest UDP, explain how you'll handle packet loss at the application layer
- Mention that modern applications often use **both** (e.g., TCP for auth, UDP for video)
- Connection overhead is why **HTTP/2 multiplexing** and **connection pooling** matter